# GraphRAG — Trial & Error Schema Linking (SLA) + EDA

Notebook ringan untuk **utak-atik retrieval GraphRAG sampai SLA-nya bagus**, tanpa LLM dan tanpa full Spider. Hanya sampel kecil dev set (default 15%).

| Bagian | Isi |
|---|---|
| 1-3 | Setup, konfigurasi (semua knob ada di sel 3), load data + sampel + gold schema |
| 4 | **EDA**: struktur graph, ukuran gold schema, efek normalisasi nama, nama kolom ambigu |
| 5 | Bangun index embedding (cache, jalan sekali) |
| 6 | **Walk-through 1 query**: skor Stage 1 / Stage 2, node tambahan dari traversal, prediksi vs gold |
| 7 | **SLA** semua sampel (table-level & column-level, macro-average, `src/metrics/sla.py`) |
| 8 | **Error analysis**: kenapa kolom gold terlewat (tabel salah / di bawah threshold / terpotong cap / ...) |
| 9 | **Mini-grid** `tables_pct x columns_pct x threshold` + heatmap |

**Konsisten dengan `sweep.py`:** sampel diambil dengan `np.random.default_rng(cfg.seed)` yang sama, gold schema dari `extract_ground_truth_schema()` resmi, retrieval lewat `retrieve_graphrag_schema()` yang sama dengan pipeline. Jadi dengan `SAMPLE_RATIO` yang sama, angkanya sama dengan `sweep.py --sample <ratio>`.

> Ini alat eksplorasi. Nilai final tetap dipilih lewat `sweep.py` pada sampel yang lebih besar, lalu ditulis ke `config.py`.

## 1. Setup

In [ ]:
import os, subprocess
from pathlib import Path

REPO_URL = "https://github.com/william-darma-wijaya/thesis-2803.git"
REPO_BRANCH = "feat/embed-normalization"  # branch yang di-clone di Kaggle

def sh(cmd: str) -> None:
    print(f"$ {cmd}")
    subprocess.run(cmd, shell=True, check=True)

def is_repo_root(p: Path) -> bool:
    return (p / "src" / "core" / "config.py").is_file()

if not is_repo_root(Path.cwd()):
    if is_repo_root(Path.cwd().parent):      # dijalankan dari notebooks/
        os.chdir("..")
    else:                                    # Kaggle: clone dulu
        if not is_repo_root(Path("thesis-2803").resolve()):
            sh(f"git clone -b {REPO_BRANCH} {REPO_URL}")
        os.chdir("thesis-2803")
REPO_ROOT = Path.cwd()
assert is_repo_root(REPO_ROOT), f"{REPO_ROOT} bukan root repo thesis-2803"
print("Repo root:", REPO_ROOT)

# nltk dibutuhkan parser SQL Spider (process_sql.py); sentence-transformers untuk BGE-M3
sh("pip install -q sentence-transformers nltk")
import nltk
for pkg in ("punkt", "punkt_tab"):
    try:
        nltk.download(pkg, quiet=True)
    except Exception as e:
        print(f"[WARN] nltk.download({pkg}) gagal: {e}")

## 2. Import

In [ ]:
import sys, json, re, math
from collections import Counter
from dataclasses import replace
from itertools import product

import numpy as np
import pandas as pd
import networkx as nx
import torch
import matplotlib.pyplot as plt
from IPython.display import display
from sentence_transformers import SentenceTransformer, util
from tqdm.auto import tqdm

sys.path.insert(0, str(REPO_ROOT))

from src.core.config import PipelineConfig
from src.core.schema import load_spider_schema, build_schema_graph, normalize_identifier
from src.retrieval.retrieval import (
    build_schema_index, retrieve_candidate_tables, retrieve_graphrag_schema,
    column_nodes_to_schema, trace_schema_paths, _extract_phrases, k_from_pct,
)
from src.retrieval.baseline import (
    build_table_graph, build_table_index, retrieve_baseline_tables, table_nodes_to_schema,
)
from src.metrics.sla import compute_sla, aggregate_sla, extract_ground_truth_schema
from src.utils.schema_utils import load_db_schema

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 200)
print("Imports OK | CUDA:", torch.cuda.is_available())

## 3. Konfigurasi — semua knob ada di sini

Ubah nilai di sel ini, lalu jalankan ulang dari bagian 7 (SLA) ke bawah. Index embedding (bagian 5) **tidak perlu dibangun ulang** kalau yang diubah hanya top-k / threshold / n-gram.
Kalau mengubah cara teks di-embed, ubah `EMBED_TABLE_PREFIX` di bagian 5 (atau `normalize_identifier`, lalu restart kernel) dan bangun ulang index.

| Knob | Arti |
|---|---|
| `SAMPLE_RATIO` | fraksi dev set (0.10 - 0.20 cukup untuk trial) |
| `TABLES_PCT` | Stage 1: simpan `ceil(pct x #tabel DB)` tabel teratas (`0` = semua tabel) |
| `COLUMNS_PCT` | Stage 2: cap `ceil(pct x #kolom di tabel kandidat)` (`0` = tanpa cap) |
| `THRESHOLD` | skor cosine minimum (max lintas n-gram) agar kolom dipertahankan |
| `MAX_NGRAM` | query dipecah jadi n-gram sampai panjang ini |
| `EXCLUDE_EMPTY_GOLD` | buang query yang gold schema-nya kosong (mis. `SELECT count(*) FROM t`). `sweep.py` TIDAK membuangnya, jadi biarkan `False` kalau mau angka sama |

In [ ]:
SAMPLE_RATIO = 0.15
TABLES_PCT   = 0.6
COLUMNS_PCT  = 0.6
THRESHOLD    = 0.35
MAX_NGRAM    = 3
EXCLUDE_EMPTY_GOLD = False
RUN_BASELINE = True          # hitung SLA Baseline sebagai pembanding (sel 7)

cfg = PipelineConfig(
    top_k_tables_pct=TABLES_PCT,
    top_k_columns_pct=COLUMNS_PCT,
    semantic_similarity_threshold=THRESHOLD,
    max_ngram=MAX_NGRAM,
)

# Lokasi data Spider: Kaggle (default config) -> data/spider_data lokal -> env SPIDER_DATA_PATH
_candidates = [
    os.environ.get("SPIDER_DATA_PATH"),
    cfg.data_path,
    REPO_ROOT / "data" / "spider_data",
    REPO_ROOT / "data" / "spider_data" / "spider_data",
]
for _p in _candidates:
    if _p and (Path(_p) / "tables.json").exists():
        cfg.data_path = Path(_p)
        break
else:
    raise FileNotFoundError("tables.json tidak ketemu. Set SPIDER_DATA_PATH atau taruh dataset di data/spider_data/")
print("Data path:", cfg.data_path)

## 3b. Load data, graph, sampel dev set, dan gold schema

In [ ]:
schema_df = load_spider_schema(cfg.tables_json)
graph     = build_schema_graph(schema_df)        # node = kolom (GraphRAG)
tbl_graph = build_table_graph(schema_df)         # node = tabel (Baseline)
db_schema = load_db_schema(str(cfg.tables_json))

with open(cfg.dev_json, "r", encoding="utf-8") as f:
    dev_data = json.load(f)

# Sampel IDENTIK dengan sweep.py
n = max(1, int(len(dev_data) * SAMPLE_RATIO))
rng = np.random.default_rng(cfg.seed)
indices = sorted(rng.choice(len(dev_data), size=n, replace=False))

samples, n_failed, n_empty = [], 0, 0      # samples: list of (query_id, item, gold_set)
for i in indices:
    item = dev_data[i]
    try:
        gold = extract_ground_truth_schema(item["query"], item["db_id"], db_schema)
    except Exception:
        n_failed += 1
        continue
    if not gold:
        n_empty += 1
        if EXCLUDE_EMPTY_GOLD:
            continue
    samples.append((i, item, gold))

sample_dbs = list(dict.fromkeys(item["db_id"] for _, item, _ in samples))
print(f"Dev set: {len(dev_data)} query | sampel: {n} | dipakai: {len(samples)} "
      f"| gold gagal di-parse: {n_failed} | gold kosong: {n_empty}")
print(f"Database di sampel: {len(sample_dbs)} dari {schema_df['Database'].nunique()}")

## 4. EDA singkat

Empat pertanyaan yang dijawab di sini:
1. **Seperti apa graph-nya?** (ukuran per DB, jenis edge, komponen terpisah)
2. **Seberapa kecil gold schema?** Ini batas atas seberapa selektif retrieval boleh dibuat.
3. **Apa yang berubah setelah normalisasi nama?** (teks yang benar-benar di-embed)
4. **Nama kolom mana yang ambigu?** (`id`, `name` ada di banyak tabel, penyebab utama presisi turun)

### 4a. Struktur schema graph

In [ ]:
rows = []
for db in schema_df["Database"].unique():
    sub = graph.subgraph([n for n, d in graph.nodes(data=True) if d["database"] == db])
    tables = {d["table"] for _, d in sub.nodes(data=True)}
    rows.append({
        "db": db, "tables": len(tables), "columns": sub.number_of_nodes(), "edges": sub.number_of_edges(),
        "components": nx.number_connected_components(sub),
    })
db_stats = pd.DataFrame(rows).set_index("db")
db_stats["in_sample"] = db_stats.index.isin(sample_dbs)

rel_counts = Counter(d.get("relation") for _, _, d in graph.edges(data=True))
print(f"Graph: {graph.number_of_nodes()} node (kolom), {graph.number_of_edges()} edge")
print("Edge per tipe:", dict(rel_counts))
print(f"DB dengan >1 komponen terpisah: {(db_stats['components'] > db_stats['tables']).sum()} "
      "(komponen > tabel = ada kolom/tabel yang tidak terhubung ke PK, traversal tidak bisa menyambungnya)")

fig, ax = plt.subplots(1, 3, figsize=(15, 3.6))
ax[0].hist(db_stats["tables"], bins=range(1, int(db_stats["tables"].max()) + 2), edgecolor="white")
ax[0].set(title="Tabel per database", xlabel="# tabel", ylabel="# database")
ax[1].hist(db_stats["columns"], bins=20, edgecolor="white")
ax[1].set(title="Kolom per database", xlabel="# kolom")
deg = [d for _, d in graph.degree()]
ax[2].hist(deg, bins=range(0, max(deg) + 2), edgecolor="white")
ax[2].set(title="Degree node kolom", xlabel="degree", yscale="log")
plt.tight_layout(); plt.show()
display(db_stats.sort_values("columns", ascending=False).head(8))

### 4b. Ukuran gold schema vs ukuran database

In [ ]:
gold_rows = []
for qid, item, gold in samples:
    db = item["db_id"]
    n_db_cols = len(db_schema[db])
    gold_rows.append({
        "qid": qid, "db_id": db,
        "gold_tables": len({g.split(".", 1)[0] for g in gold}),
        "gold_cols": len(gold),
        "db_tables": len({c.split(".", 1)[0] for c in db_schema[db]}),
        "db_cols": n_db_cols,
        "gold_ratio": len(gold) / n_db_cols,
    })
gold_df = pd.DataFrame(gold_rows)
print(gold_df[["gold_tables", "gold_cols", "db_tables", "db_cols", "gold_ratio"]].describe().round(2).T[["mean", "50%", "min", "max"]])
print(f"\nRata-rata gold memakai {gold_df['gold_ratio'].mean():.1%} kolom DB -> retrieval ideal hanya perlu mengirim sebagian kecil schema.")

fig, ax = plt.subplots(1, 3, figsize=(15, 3.6))
ax[0].hist(gold_df["gold_tables"], bins=range(0, gold_df["gold_tables"].max() + 2), edgecolor="white")
ax[0].set(title="Tabel yang dipakai gold SQL", xlabel="# tabel", ylabel="# query")
ax[1].hist(gold_df["gold_cols"], bins=range(0, gold_df["gold_cols"].max() + 2), edgecolor="white")
ax[1].set(title="Kolom yang dipakai gold SQL", xlabel="# kolom")
ax[2].scatter(gold_df["db_cols"], gold_df["gold_cols"], alpha=0.4)
ax[2].set(title="Gold cols vs ukuran DB", xlabel="# kolom DB", ylabel="# kolom gold")
plt.tight_layout(); plt.show()

### 4c. Efek normalisasi nama (teks yang di-embed)

Embedding memakai `normalize_identifier()` (`src/core/schema.py`): `_` jadi spasi, camelCase dipecah, lowercase. Di bawah: contoh sebelum/sesudah, dan nama yang **masih** terlihat seperti singkatan (kandidat masalah karena BGE-M3 tidak bisa menebak `stu` = `student`).

In [ ]:
cols_all = schema_df[schema_df["Column"] != "*"][["Database", "Table", "Column"]].drop_duplicates()
cols_all["embedded_text"] = [f"{normalize_identifier(t)} {normalize_identifier(c)}"
                             for t, c in zip(cols_all["Table"], cols_all["Column"])]
changed = cols_all[cols_all["Column"].str.lower() != cols_all["Column"].map(normalize_identifier)]
print(f"Nama kolom yang berubah oleh normalisasi: {len(changed)} / {len(cols_all)} ({len(changed)/len(cols_all):.0%})")
display(changed.sample(12, random_state=cfg.seed)[["Table", "Column", "embedded_text"]].reset_index(drop=True))

# Heuristik kasar: token hasil normalisasi yang sangat pendek (<=3 huruf) dan bukan kata umum -> kemungkinan singkatan
COMMON_SHORT = {"id", "no", "the", "of", "and", "for", "age", "age", "key", "ref", "url", "tel", "dob", "sex", "day",
                "end", "fee", "net", "due", "tax", "age", "bed", "map", "set", "top", "low", "max", "min", "avg",
                "sum", "num", "per", "pct", "ssn", "gpa", "zip", "net", "win", "won", "lost", "loss", "tie", "pts"}
tok = Counter()
for c in cols_all["Column"].map(normalize_identifier):
    for w in c.split():
        if w.isalpha() and len(w) <= 3 and w not in COMMON_SHORT:
            tok[w] += 1
print("\nToken pendek yang paling sering muncul (cek apakah singkatan):")
print(", ".join(f"{w}({n})" for w, n in tok.most_common(25)))

### 4d. Nama kolom ambigu (muncul di banyak tabel)

In [ ]:
amb = (cols_all.assign(col_l=cols_all["Column"].str.lower())
       .groupby(["Database", "col_l"])["Table"].nunique().reset_index(name="n_tables"))
amb = amb[amb["n_tables"] > 1]
print(f"Pasangan (DB, nama kolom) yang ada di >1 tabel: {len(amb)}")
top_names = amb.groupby("col_l")["Database"].count().sort_values(ascending=False).head(12)
print("Nama kolom paling sering ambigu:", dict(top_names))
print("-> Karena itu embedding kolom memakai 'tabel kolom', dan traversal memakai node id (db.table.column), bukan nama kolom saja.")

## 5. Bangun index embedding

`build_schema_index()` meng-embed semua kolom dan tabel per database (hanya DB yang ada di sampel). Jalan **sekali**.
`CachedEmbedder` membungkus model supaya n-gram query yang sama tidak di-encode ulang tiap kali knob diubah, jadi grid di bagian 9 cepat.

In [ ]:
class CachedEmbedder:
    """Drop-in untuk SentenceTransformer.encode(): memoize hasil per daftar teks."""
    def __init__(self, model):
        self.model, self._cache = model, {}
    def encode(self, texts, **kw):
        key = (tuple(texts), tuple(sorted(kw.items())))
        if key not in self._cache:
            self._cache[key] = self.model.encode(texts, **kw)
        return self._cache[key]

print("Loading embedding model:", cfg.embedding_model)
_model = SentenceTransformer(cfg.embedding_model)
embedder = CachedEmbedder(_model)

# False = teks embed kolom hanya nama kolom ("song release year"); True = "singer song release year" (lama).
# Untuk A/B: ubah flag lalu jalankan ulang sel ini dan bagian 6-9.
EMBED_TABLE_PREFIX = False
col_cache = {db: build_schema_index(graph, db, _model, include_table_in_text=EMBED_TABLE_PREFIX)
             for db in tqdm(sample_dbs, desc="Index kolom (GraphRAG)")}
n_cols_db = {db: len(db_schema[db]) for db in sample_dbs}
print("Selesai.")

## 6. Walk-through 1 query

Lihat apa yang dilakukan tiap tahap untuk satu pertanyaan. Ganti `QUERY_POS` (posisi di daftar sampel) untuk query lain.
- **Stage 1**: skor tiap tabel (max cosine lintas n-gram), top-k jadi kandidat.
- **Stage 2**: skor tiap kolom di tabel kandidat; lolos threshold, lalu di-cap.
- **Traversal**: node tambahan di jalur terpendek antar kolom terpilih (untuk JOIN).

In [ ]:
def stage_scores(question, db_id, cfg):
    """Replikasi skor Stage 1 & 2 (sama dengan retrieval.py) untuk keperluan inspeksi."""
    idx = col_cache[db_id]
    phrases = _extract_phrases(question, cfg.max_ngram)
    phrase_emb = embedder.encode(phrases, convert_to_tensor=True)

    t_scores = util.cos_sim(phrase_emb, idx.table_embeddings).max(dim=0).values
    cand = retrieve_candidate_tables(question, idx, embedder, cfg)

    cand_set = set(cand)
    mask = [t in cand_set for t in idx.col_table_map]
    ids = [n for n, m in zip(idx.col_node_ids, mask) if m]
    c_scores = util.cos_sim(phrase_emb, idx.col_embeddings[torch.tensor(mask)]).max(dim=0).values
    return idx, cand, t_scores, ids, c_scores

def short(node):  # "db.table.column" -> "table.column" (lowercase, format SLA)
    d = graph.nodes[node]
    return f"{d['table'].lower()}.{d['column'].lower()}"

QUERY_POS = 0
qid, item, gold = samples[QUERY_POS]
q, db = item["question"], item["db_id"]
idx, cand, t_scores, ids, c_scores = stage_scores(q, db, cfg)

print(f"[{QUERY_POS}] db={db}\nQ    : {q}\nSQL  : {item['query']}\nGold : {sorted(gold)}\n")

k_t = k_from_pct(cfg.top_k_tables_pct, len(idx.tables))
print(f"Stage 1 — top {k_t} dari {len(idx.tables)} tabel:")
for r, i in enumerate(torch.argsort(t_scores, descending=True).tolist()):
    t = idx.tables[i]
    mark = "KANDIDAT" if r < k_t else "        "
    gold_mark = "  <- gold" if any(g.startswith(t.lower() + ".") for g in gold) else ""
    print(f"  {mark} {t_scores[i].item():.3f}  {t}{gold_mark}")

k_c = k_from_pct(cfg.top_k_columns_pct, len(ids))
order = torch.argsort(c_scores, descending=True).tolist()
print(f"\nStage 2 — {len(ids)} kolom di tabel kandidat, cap={k_c}, threshold={cfg.semantic_similarity_threshold}:")
for r, i in enumerate(order[:15]):
    s = c_scores[i].item()
    keep = s >= cfg.semantic_similarity_threshold and r < k_c
    print(f"  {'KEEP' if keep else '    '} {s:.3f}  {short(ids[i])}{'  <- gold' if short(ids[i]) in gold else ''}")

column_nodes, _ = retrieve_graphrag_schema(graph, db, q, idx, embedder, cfg)
pred = set(column_nodes_to_schema(graph, column_nodes))
detected = {short(ids[i]) for i in order[:k_c] if c_scores[i].item() >= cfg.semantic_similarity_threshold}
print(f"\nDitambah traversal : {sorted(pred - detected)}")
print(f"Prediksi final     : {sorted(pred)}")
print(f"Terlewat (FN)      : {sorted(gold - pred)}")
print(f"Berlebih (FP)      : {sorted(pred - gold)}")
for lvl in ("table", "column"):
    r = compute_sla(gold, pred, level=lvl)
    print(f"SLA {lvl:<6}: P={r.precision:.2f} R={r.recall:.2f} F1={r.f1:.2f}")

## 7. SLA atas seluruh sampel

`run_graphrag()` menjalankan retrieval GraphRAG per query dan menghitung SLA per query (`compute_sla`), lalu rata-rata macro (`aggregate_sla`), persis seperti `sweep.py`. Hasil per query disimpan di DataFrame untuk dianalisis di bagian 8.

In [ ]:
def _summarize(per_query):
    t = aggregate_sla([r["_t"] for r in per_query])
    c = aggregate_sla([r["_c"] for r in per_query])
    return {
        "tbl_P": t.precision, "tbl_R": t.recall, "tbl_F1": t.f1,
        "col_P": c.precision, "col_R": c.recall, "col_F1": c.f1,
        "cols_sent": np.mean([r["cols_sent"] for r in per_query]),
        "cols_db": np.mean([r["cols_db"] for r in per_query]),
        "n": len(per_query),
    }

def _row(qid, item, gold, pred):
    pred = set(pred)
    return {
        "qid": qid, "db_id": item["db_id"], "question": item["question"],
        "gold": sorted(gold), "pred": sorted(pred),
        "tbl_F1": compute_sla(gold, pred, "table").f1, "col_P": compute_sla(gold, pred, "column").precision,
        "col_R": compute_sla(gold, pred, "column").recall, "col_F1": compute_sla(gold, pred, "column").f1,
        "cols_sent": len(pred), "cols_db": n_cols_db[item["db_id"]],
        "_t": compute_sla(gold, pred, "table"), "_c": compute_sla(gold, pred, "column"),
    }

def run_graphrag(cfg, samples=samples):
    out = []
    for qid, item, gold in samples:
        db = item["db_id"]
        nodes, _ = retrieve_graphrag_schema(graph, db, item["question"], col_cache[db], embedder, cfg)
        out.append(_row(qid, item, gold, column_nodes_to_schema(graph, nodes)))
    return out

res = run_graphrag(cfg)
per_query = pd.DataFrame(res).drop(columns=["_t", "_c"])
g = _summarize(res)

summary = {"GraphRAG": g}
if RUN_BASELINE:
    tbl_cache = {db: build_table_index(tbl_graph, db, _model) for db in tqdm(sample_dbs, desc="Index tabel (Baseline)")}
    base = []
    for qid, item, gold in samples:
        db = item["db_id"]
        nodes = retrieve_baseline_tables(tbl_graph, db, item["question"], tbl_cache[db], embedder, cfg)
        base.append(_row(qid, item, gold, table_nodes_to_schema(tbl_graph, nodes)))
    summary["Baseline"] = _summarize(base)

tbl = pd.DataFrame(summary).T
tbl["col_reduction"] = 1 - tbl["cols_sent"] / tbl["cols_db"]
print(f"Config: tables_pct={cfg.top_k_tables_pct} columns_pct={cfg.top_k_columns_pct} "
      f"threshold={cfg.semantic_similarity_threshold} max_ngram={cfg.max_ngram} | n={g['n']}")
display(tbl.style.format({
    "tbl_P": "{:.1%}", "tbl_R": "{:.1%}", "tbl_F1": "{:.1%}", "col_P": "{:.1%}", "col_R": "{:.1%}",
    "col_F1": "{:.1%}", "cols_sent": "{:.1f}", "cols_db": "{:.1f}", "col_reduction": "{:.1%}", "n": "{:.0f}",
}))
print("cols_sent = kolom yang masuk prompt | col_reduction = penghematan kolom vs mengirim seluruh DB")

## 8. Error analysis — kenapa kolom gold terlewat?

Setiap kolom gold yang **tidak** ada di prediksi diberi satu alasan, diperiksa berurutan:

| Alasan | Arti | Knob yang relevan |
|---|---|---|
| `stage1_table_miss` | tabelnya sendiri tidak masuk top-k Stage 1 | `TABLES_PCT` naik, atau teks embed tabel kurang informatif |
| `below_threshold` | tabel lolos, tapi skor kolom < threshold | `THRESHOLD` turun, atau nama kolom sulit dipahami embedder |
| `cut_by_cap` | lolos threshold, tapi peringkatnya di luar cap | `COLUMNS_PCT` naik |
| `other` | tidak masuk walau lolos semua (mis. fallback / kasus tepi) | cek manual |

Kolom gold yang **berhasil** masuk dibedakan: `hit_stage2` (dipilih langsung) vs `hit_via_traversal` (hanya karena ada di jalur JOIN).

In [ ]:
def diagnose(cfg, samples=samples):
    rows = []
    for qid, item, gold in samples:
        db, q = item["db_id"], item["question"]
        idx, cand, _, ids, c_scores = stage_scores(q, db, cfg)
        cand_l = {t.lower() for t in cand}
        order = torch.argsort(c_scores, descending=True).tolist()
        k_c = k_from_pct(cfg.top_k_columns_pct, len(ids))
        rank = {short(ids[i]): r for r, i in enumerate(order)}
        score = {short(ids[i]): c_scores[i].item() for i in range(len(ids))}
        nodes, _ = retrieve_graphrag_schema(graph, db, q, col_cache[db], embedder, cfg)
        pred = set(column_nodes_to_schema(graph, nodes))
        detected = {c for c, r in rank.items() if r < k_c and score[c] >= cfg.semantic_similarity_threshold}
        for g_col in gold:
            t = g_col.split(".", 1)[0]
            if g_col in pred:
                reason = "hit_stage2" if g_col in detected else "hit_via_traversal"
            elif t not in cand_l:
                reason = "stage1_table_miss"
            elif score.get(g_col, -1) < cfg.semantic_similarity_threshold:
                reason = "below_threshold"
            elif rank.get(g_col, 10**9) >= k_c:
                reason = "cut_by_cap"
            else:
                reason = "other"
            rows.append({"qid": qid, "db_id": db, "question": q, "gold_col": g_col, "reason": reason,
                         "score": score.get(g_col), "rank": rank.get(g_col), "cap": k_c})
    return pd.DataFrame(rows)

diag = diagnose(cfg)
order_reasons = ["hit_stage2", "hit_via_traversal", "below_threshold", "cut_by_cap", "stage1_table_miss", "other"]
counts = diag["reason"].value_counts().reindex(order_reasons, fill_value=0)
print(f"Total kolom gold: {len(diag)}")
display(pd.DataFrame({"jumlah": counts, "persen": (counts / len(diag)).map("{:.1%}".format)}))

fig, ax = plt.subplots(1, 3, figsize=(16, 3.8))
ax[0].barh(counts.index[::-1], counts.values[::-1])
ax[0].set(title="Nasib kolom gold", xlabel="# kolom")
ax[1].hist(per_query["col_F1"], bins=10, range=(0, 1), edgecolor="white")
ax[1].set(title="Distribusi col F1 per query", xlabel="col F1", ylabel="# query")
ax[2].scatter(per_query["cols_sent"], per_query["col_R"], alpha=0.4)
ax[2].set(title="Recall vs kolom terkirim", xlabel="# kolom di prompt", ylabel="col recall")
plt.tight_layout(); plt.show()

misses = diag[~diag["reason"].str.startswith("hit")]
print("Skor kolom gold yang terlewat karena below_threshold (dekat threshold = cukup turunkan threshold):")
display(misses[misses["reason"] == "below_threshold"]["score"].describe().round(3).to_frame().T)

Query dengan F1 kolom terendah, untuk dilihat manual (apa yang terlewat, apa yang berlebih):

In [ ]:
worst = per_query.sort_values(["col_F1", "col_R"]).head(10).copy()
worst["missed"] = [sorted(set(g) - set(p)) for g, p in zip(worst["gold"], worst["pred"])]
worst["extra"] = [sorted(set(p) - set(g)) for g, p in zip(worst["gold"], worst["pred"])]
display(worst[["qid", "db_id", "question", "col_F1", "missed", "extra"]].reset_index(drop=True))

Detail alasan untuk kolom gold yang terlewat (urut dari skor tertinggi = paling dekat lolos):

In [ ]:
display(misses.sort_values("score", ascending=False)
        [["qid", "db_id", "gold_col", "reason", "score", "rank", "cap", "question"]].head(15).reset_index(drop=True))

## 9. Mini-grid

Grid kecil di sampel ini untuk merasakan sensitivitas knob (embedding sudah di-cache, jadi cepat). Ini **bukan** pengganti `sweep.py` untuk memilih nilai final.
Ubah list di bawah. Baris diurutkan dari `col_F1` tertinggi; tie-break: kolom terkirim paling sedikit (sama seperti `sweep.py`).

In [ ]:
GRID_TABLES     = [0.4, 0.6, 0.8]
GRID_COLUMNS    = [0.4, 0.6, 0.8]
GRID_THRESHOLDS = [0.4, 0.5, 0.6, 0.7]

grid_rows = []
for tp, cp, th in tqdm(list(product(GRID_TABLES, GRID_COLUMNS, GRID_THRESHOLDS)), desc="Grid"):
    c = replace(cfg, top_k_tables_pct=tp, top_k_columns_pct=cp, semantic_similarity_threshold=th)
    s = _summarize(run_graphrag(c))
    grid_rows.append({"tables_pct": tp, "columns_pct": cp, "threshold": th, **s})
grid = pd.DataFrame(grid_rows)
grid_sorted = grid.sort_values(["col_F1", "cols_sent"], ascending=[False, True]).reset_index(drop=True)
display(grid_sorted[["tables_pct", "columns_pct", "threshold", "tbl_F1", "col_P", "col_R", "col_F1", "cols_sent"]]
        .head(10).style.format({"tbl_F1": "{:.1%}", "col_P": "{:.1%}", "col_R": "{:.1%}", "col_F1": "{:.1%}", "cols_sent": "{:.1f}"}))

fig, axes = plt.subplots(1, len(GRID_THRESHOLDS), figsize=(5.2 * len(GRID_THRESHOLDS), 4), squeeze=False)
for ax, th in zip(axes[0], GRID_THRESHOLDS):
    p = grid[grid["threshold"] == th].pivot(index="tables_pct", columns="columns_pct", values="col_F1")
    im = ax.imshow(p.values, cmap="viridis", origin="lower", aspect="auto")
    ax.set_xticks(range(len(p.columns)), [f"{c:g}" for c in p.columns])
    ax.set_yticks(range(len(p.index)), [f"{i:g}" for i in p.index])
    ax.set(title=f"col F1 (threshold={th})", xlabel="columns_pct", ylabel="tables_pct")
    for (r, c_), v in np.ndenumerate(p.values):
        ax.text(c_, r, f"{v:.2f}", ha="center", va="center", color="white", fontsize=9)
plt.tight_layout(); plt.show()

### Cara memakai hasilnya

1. Lihat **bagian 8**: alasan terlewat yang dominan menentukan knob mana yang layak diubah (`below_threshold` -> threshold / teks embed, `cut_by_cap` -> `COLUMNS_PCT`, `stage1_table_miss` -> `TABLES_PCT`).
2. Ubah knob di **sel 3**, jalankan ulang bagian 7-9.
3. Kalau sudah ada kandidat, validasi dengan `python src/experiments/sweep.py --sample 0.2 --thresholds ...`, lalu tulis pemenang ke `config.py`.